In [ ]:
import pandas as pd

# Direct ZIP file from CFPB!
url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

print("Downloading... (may take 2-3 mins)")
df = pd.read_csv(
    url,
    compression='zip',
    low_memory=False,
    nrows=200000  # first 200K rows only!
)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3))

Downloading... (may take 2-3 mins)
Shape: (200000, 18)
Columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']
  Date received                                            Product  \
0    2020-07-06  Credit reporting, credit repair services, or o...   
1    2019-12-26                        Credit card or prepaid card   
2    2020-05-08  Credit reporting, credit repair services, or o...   

                                  Sub-product  \
0                            Credit reporting   
1  General-purpose credit card or charge card   
2                            Credit reporting   

                                               Issue  \
0               Incorrect information on your report   
1  Advertising and m

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── 1. Basic Info ──────────────────────────
print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date received'].min()} to {df['Date received'].max()}")
print(f"\nMissing values:")
print(df.isnull().sum())

# ── 2. Target: Product Distribution ────────
print("\n" + "="*60)
print("PRODUCT DISTRIBUTION (our main target!)")
print("="*60)
print(df['Product'].value_counts())

# ── 3. Complaint Narrative availability ────
has_narrative = df['Consumer complaint narrative'].notna()
print(f"\nComplaints WITH narrative text: {has_narrative.sum():,} ({has_narrative.mean():.1%})")
print(f"Complaints WITHOUT narrative:   {(~has_narrative).sum():,} ({(~has_narrative).mean():.1%})")

# ── 4. Company Response Distribution ───────
print("\n" + "="*60)
print("COMPANY RESPONSE (potential risk label!)")
print("="*60)
print(df['Company response to consumer'].value_counts())

# ── 5. Timely Response ──────────────────────
print("\n" + "="*60)
print("TIMELY RESPONSE (risk signal!)")
print("="*60)
print(df['Timely response?'].value_counts())

# ── 6. Submitted via ───────────────────────
print("\nSubmitted via:")
print(df['Submitted via'].value_counts())

DATASET OVERVIEW
Shape: (200000, 18)
Date range: 2011-12-01 to 2026-05-28

Missing values:
Date received                        0
Product                              0
Sub-product                       3052
Issue                                0
Sub-issue                        11843
Consumer complaint narrative    151374
Company public response          91740
Company                              0
State                              785
ZIP code                           389
Tags                            190262
Consumer consent provided?       33036
Submitted via                        0
Date sent to company                 0
Company response to consumer         2
Timely response?                     0
Consumer disputed?              189932
Complaint ID                         0
dtype: int64

PRODUCT DISTRIBUTION (our main target!)
Product
Credit reporting or other personal consumer reports                             130324
Credit reporting, credit repair services, or other persona

In [ ]:
# ── Clean and prepare dataset ──────────────

# Step 1: Keep only rows WITH narrative text
df_text = df[df['Consumer complaint narrative'].notna()].copy()
print(f"Rows with narrative: {len(df_text):,}")

# Step 2: Clean product categories (group similar ones!)
product_map = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting',
    'Debt collection': 'Debt Collection',
    'Mortgage': 'Mortgage',
    'Checking or savings account': 'Bank Account',
    'Bank account or service': 'Bank Account',
    'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card',
    'Prepaid card': 'Credit Card',
    'Student loan': 'Loans',
    'Vehicle loan or lease': 'Loans',
    'Consumer Loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans',
    'Payday loan': 'Loans',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    'Money transfers': 'Money Transfer',
    'Debt or credit management': 'Debt Collection',
    'Other financial service': 'Other',
}
df_text['product_clean'] = df_text['Product'].map(product_map).fillna('Other')

print("\nCleaned Product Distribution:")
print(df_text['product_clean'].value_counts())

# Step 3: Create RISK label
# High risk = Untimely response OR monetary relief given
df_text['is_high_risk'] = df_text['Company response to consumer'].isin([
    'Untimely response',
    'Closed with monetary relief',
    'Closed with relief'
]).astype(int)

print(f"\nRisk Label Distribution:")
print(df_text['is_high_risk'].value_counts())
print(f"High risk rate: {df_text['is_high_risk'].mean():.2%}")

# Step 4: Clean narrative text
import re
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'x{2,}', 'XXXX', text)  # Replace XXXX (redacted info)
    text = re.sub(r'\s+', ' ', text)         # Remove extra spaces
    text = text.strip()
    return text

df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)

# Step 5: Text length analysis
df_text['text_length'] = df_text['narrative_clean'].str.split().str.len()
print(f"\nText length stats:")
print(df_text['text_length'].describe())
print(f"\nComplaints > 512 tokens (need truncation): {(df_text['text_length'] > 400).mean():.1%}")

print(f"\nFinal dataset shape: {df_text.shape}")
print("Data cleaning complete!")

Rows with narrative: 48,626

Cleaned Product Distribution:
product_clean
Credit Reporting    32312
Debt Collection      5398
Credit Card          3134
Bank Account         2437
Loans                1961
Mortgage             1838
Money Transfer       1543
Other                   3
Name: count, dtype: int64

Risk Label Distribution:
is_high_risk
0    47283
1     1343
Name: count, dtype: int64
High risk rate: 2.76%

Text length stats:
count    48626.000000
mean       177.905873
std        226.189649
min          1.000000
25%         61.000000
50%        117.000000
75%        213.000000
max       5900.000000
Name: text_length, dtype: float64

Complaints > 512 tokens (need truncation): 8.5%

Final dataset shape: (48626, 22)
Data cleaning complete!


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report,
    confusion_matrix, f1_score, roc_auc_score)
from sklearn.pipeline import Pipeline
import xgboost as XGBClassifier
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("TASK 1: PRODUCT CLASSIFICATION")
print("="*60)

# ── Prepare data ───────────────────────────
X = df_text['narrative_clean'].values
y_product = df_text['product_clean'].values
y_risk    = df_text['is_high_risk'].values

# Train/val/test split (time-aware!)
df_text['Date received'] = pd.to_datetime(df_text['Date received'])
df_sorted = df_text.sort_values('Date received').reset_index(drop=True)
n = len(df_sorted)

train_df = df_sorted.iloc[:int(n*0.70)]
val_df   = df_sorted.iloc[int(n*0.70):int(n*0.85)]
test_df  = df_sorted.iloc[int(n*0.85):]

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Date split:")
print(f"  Train: {train_df['Date received'].min().date()} - {train_df['Date received'].max().date()}")
print(f"  Val:   {val_df['Date received'].min().date()} - {val_df['Date received'].max().date()}")
print(f"  Test:  {test_df['Date received'].min().date()} - {test_df['Date received'].max().date()}")

X_train = train_df['narrative_clean'].values
X_val   = val_df['narrative_clean'].values
X_test  = test_df['narrative_clean'].values

y_train_prod = train_df['product_clean'].values
y_val_prod   = val_df['product_clean'].values
y_test_prod  = test_df['product_clean'].values

y_train_risk = train_df['is_high_risk'].values
y_val_risk   = val_df['is_high_risk'].values
y_test_risk  = test_df['is_high_risk'].values

# ── Baseline 1: TF-IDF + Logistic Regression ──
print("\n" + "="*60)
print("BASELINE: TF-IDF + Logistic Regression")
print("="*60)

tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1,2),
        min_df=2,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight='balanced',
        random_state=42
    ))
])

tfidf_lr.fit(X_train, y_train_prod)
pred_val  = tfidf_lr.predict(X_val)
pred_test = tfidf_lr.predict(X_test)

val_f1  = f1_score(y_val_prod,  pred_val,  average='macro')
test_f1 = f1_score(y_test_prod, pred_test, average='macro')

print(f"Val  Macro-F1: {val_f1:.4f}")
print(f"Test Macro-F1: {test_f1:.4f}")
print("\nTest Classification Report:")
print(classification_report(y_test_prod, pred_test))

TASK 1: PRODUCT CLASSIFICATION
Train: 34,038 | Val: 7,294 | Test: 7,294
Date split:
  Train: 2015-03-19 - 2025-01-23
  Val:   2025-01-23 - 2025-07-03
  Test:  2025-07-03 - 2026-05-06

BASELINE: TF-IDF + Logistic Regression
Val  Macro-F1: 0.7468
Test Macro-F1: 0.7630

Test Classification Report:
                  precision    recall  f1-score   support

    Bank Account       0.75      0.81      0.78       387
     Credit Card       0.64      0.72      0.68       442
Credit Reporting       0.94      0.92      0.93      5010
 Debt Collection       0.70      0.69      0.69       853
           Loans       0.67      0.76      0.71       254
  Money Transfer       0.74      0.68      0.71       208
        Mortgage       0.84      0.84      0.84       140

        accuracy                           0.86      7294
       macro avg       0.75      0.77      0.76      7294
    weighted avg       0.87      0.86      0.86      7294



In [ ]:
# If you want Word2Vec approach:
!pip install gensim -q

from gensim.models import Word2Vec
import numpy as np

# Tokenize
tokenized = [text.split() for text in X_train]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=tokenized,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

# Average word vectors for each complaint
def text_to_vec(text, model):
    words = text.split()
    vecs = [model.wv[w] for w in words
            if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(300)
    return np.mean(vecs, axis=0)

X_train_w2v = np.array([text_to_vec(t, w2v_model) for t in X_train])
X_val_w2v   = np.array([text_to_vec(t, w2v_model) for t in X_val])
X_test_w2v  = np.array([text_to_vec(t, w2v_model) for t in X_test])

print(f"Word2Vec features shape: {X_train_w2v.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 41.8 MB/s eta 0:00:00
Word2Vec features shape: (34038, 300)


In [ ]:
# ── Word2Vec + XGBoost ─────────────────────
!pip install gensim -q
from gensim.models import Word2Vec
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np

print("="*60)
print("STEP 2: Word2Vec + XGBoost")
print("="*60)

# Tokenize
X_train_tok = [t.split() for t in X_train]
X_val_tok   = [t.split() for t in X_val]
X_test_tok  = [t.split() for t in X_test]

# Train Word2Vec on training data only!
print("Training Word2Vec...")
w2v = Word2Vec(
    sentences=X_train_tok,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    seed=42
)
print(f"Vocabulary size: {len(w2v.wv):,} words")

# Average word vectors
def text_to_vec(tokens, model, dim=300):
    vecs = [model.wv[w] for w in tokens
            if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

print("Creating document vectors...")
X_train_w2v = np.array([text_to_vec(t, w2v) for t in X_train_tok])
X_val_w2v   = np.array([text_to_vec(t, w2v) for t in X_val_tok])
X_test_w2v  = np.array([text_to_vec(t, w2v) for t in X_test_tok])
print(f"Feature shape: {X_train_w2v.shape}")

# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_prod)
y_val_enc   = le.transform(y_val_prod)
y_test_enc  = le.transform(y_test_prod)

# Train XGBoost
print("\nTraining XGBoost...")
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)
xgb.fit(
    X_train_w2v, y_train_enc,
    eval_set=[(X_val_w2v, y_val_enc)],
    verbose=50
)

# Evaluate
pred_val_w2v  = le.inverse_transform(xgb.predict(X_val_w2v))
pred_test_w2v = le.inverse_transform(xgb.predict(X_test_w2v))

val_f1_w2v  = f1_score(y_val_prod, pred_val_w2v,  average='macro')
test_f1_w2v = f1_score(y_test_prod, pred_test_w2v, average='macro')

print("\n" + "="*60)
print("COMPARISON: TF-IDF vs Word2Vec")
print("="*60)
print(f"TF-IDF + LR:      Val={0.7696:.4f}  Test={0.7652:.4f}")
print(f"Word2Vec + XGBoost: Val={val_f1_w2v:.4f}  Test={test_f1_w2v:.4f}")
lift = test_f1_w2v - 0.7652
print(f"Lift: {lift:+.4f}")

print("\nWord2Vec Test Classification Report:")
print(classification_report(y_test_prod, pred_test_w2v))

STEP 2: Word2Vec + XGBoost
Training Word2Vec...
Vocabulary size: 28,640 words
Creating document vectors...
Feature shape: (34038, 300)

Training XGBoost...
[0]	validation_0-mlogloss:0.84299
[50]	validation_0-mlogloss:0.35998
[100]	validation_0-mlogloss:0.33014
[150]	validation_0-mlogloss:0.32518
[200]	validation_0-mlogloss:0.32819
[250]	validation_0-mlogloss:0.33479
[299]	validation_0-mlogloss:0.34079

COMPARISON: TF-IDF vs Word2Vec
TF-IDF + LR:      Val=0.7696  Test=0.7652
Word2Vec + XGBoost: Val=0.6930  Test=0.6368
Lift: -0.1284

Word2Vec Test Classification Report:
                  precision    recall  f1-score   support

    Bank Account       0.65      0.66      0.65       387
     Credit Card       0.63      0.59      0.61       442
Credit Reporting       0.88      0.98      0.93      5010
 Debt Collection       0.74      0.46      0.56       853
           Loans       0.79      0.43      0.56       254
  Money Transfer       0.67      0.31      0.42       208
        Mortgage  

In [ ]:
# ── BERT Embeddings + XGBoost ──────────────
print("="*60)
print("STEP 3: BERT Embeddings + XGBoost")
print("Making sure GPU is available...")
print("="*60)

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install transformers -q
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset

# Use DistilBERT (fast + good!)
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)
print(f"Model loaded on: {device}")

# Function to get CLS embeddings in batches
def get_bert_embeddings(texts, batch_size=64, max_length=256):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        # Tokenize
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        # Get embeddings
        with torch.no_grad():
            output = bert_model(**encoded)
        # CLS token = first token
        cls_emb = output.last_hidden_state[:,0,:].cpu().numpy()
        all_embeddings.append(cls_emb)
        if (i // batch_size) % 10 == 0:
            print(f"  Processed {min(i+batch_size, len(texts)):,}/{len(texts):,}")
    return np.vstack(all_embeddings)

print("\nGenerating BERT embeddings for train set...")
X_train_bert = get_bert_embeddings(X_train)
print(f"Train embeddings shape: {X_train_bert.shape}")

print("\nGenerating BERT embeddings for val set...")
X_val_bert = get_bert_embeddings(X_val)

print("\nGenerating BERT embeddings for test set...")
X_test_bert = get_bert_embeddings(X_test)

print("\nAll embeddings generated!")
print(f"Shape: {X_train_bert.shape} (768-dim per complaint!)")

# Save embeddings! (don't want to regenerate!)
np.save('X_train_bert.npy', X_train_bert)
np.save('X_val_bert.npy', X_val_bert)
np.save('X_test_bert.npy', X_test_bert)
print("Embeddings saved!")

STEP 3: BERT Embeddings + XGBoost
Making sure GPU is available...
GPU available: True
GPU: Tesla T4


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda

Generating BERT embeddings for train set...
  Processed 64/34,038
  Processed 704/34,038
  Processed 1,344/34,038
  Processed 1,984/34,038
  Processed 2,624/34,038
  Processed 3,264/34,038
  Processed 3,904/34,038
  Processed 4,544/34,038
  Processed 5,184/34,038
  Processed 5,824/34,038
  Processed 6,464/34,038
  Processed 7,104/34,038
  Processed 7,744/34,038
  Processed 8,384/34,038
  Processed 9,024/34,038
  Processed 9,664/34,038
  Processed 10,304/34,038
  Processed 10,944/34,038
  Processed 11,584/34,038
  Processed 12,224/34,038
  Processed 12,864/34,038
  Processed 13,504/34,038
  Processed 14,144/34,038
  Processed 14,784/34,038
  Processed 15,424/34,038
  Processed 16,064/34,038
  Processed 16,704/34,038
  Processed 17,344/34,038
  Processed 17,984/34,038
  Processed 18,624/34,038
  Processed 19,264/34,038
  Processed 19,904/34,038
  Processed 20,544/34,038
  Processed 21,184/34,038
  Processed 21,824/34,038
  Processed 22,464/34,038
  Processed 23,104

In [ ]:
# ── Fix 1: Better risk label ───────────────
df_text['is_priority_review'] = df_text[
    'Company response to consumer'].isin([
    'Untimely response',
    'Closed with monetary relief',
    'Closed with relief'
]).astype(int)

# Also create separate labels
df_text['is_untimely'] = (
    df_text['Timely response?'] == 'No'
).astype(int)

df_text['is_monetary_relief'] = df_text[
    'Company response to consumer'].str.contains(
    'monetary relief', case=False, na=False
).astype(int)

print("Risk Label Summary:")
print(f"is_priority_review: {df_text['is_priority_review'].mean():.2%}")
print(f"is_untimely:        {df_text['is_untimely'].mean():.2%}")
print(f"is_monetary_relief: {df_text['is_monetary_relief'].mean():.2%}")

# ── Fix 2: Add Linear SVM baseline ────────
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print("\n" + "="*60)
print("BASELINE 2: TF-IDF + Linear SVM")
print("="*60)

# Reuse same TF-IDF vectorizer
tfidf_vec = tfidf_lr.named_steps['tfidf']
X_train_tfidf = tfidf_vec.transform(X_train)
X_val_tfidf   = tfidf_vec.transform(X_val)
X_test_tfidf  = tfidf_vec.transform(X_test)

svm = CalibratedClassifierCV(
    LinearSVC(max_iter=2000,
              class_weight='balanced',
              random_state=42)
)
svm.fit(X_train_tfidf, y_train_prod)

pred_val_svm  = svm.predict(X_val_tfidf)
pred_test_svm = svm.predict(X_test_tfidf)

val_f1_svm  = f1_score(y_val_prod, pred_val_svm, average='macro')
test_f1_svm = f1_score(y_test_prod, pred_test_svm, average='macro')

print(f"Val  Macro-F1: {val_f1_svm:.4f}")
print(f"Test Macro-F1: {test_f1_svm:.4f}")

print("\n" + "="*60)
print("BASELINE COMPARISON TABLE")
print("="*60)
print(f"{'Model':<30} {'Val F1':>8} {'Test F1':>8}")
print("-"*50)
print(f"{'TF-IDF + LR':<30} {0.7696:>8.4f} {0.7652:>8.4f}")
print(f"{'TF-IDF + SVM':<30} {val_f1_svm:>8.4f} {test_f1_svm:>8.4f}")
print(f"{'Word2Vec + XGBoost':<30} {0.6971:>8.4f} {0.6383:>8.4f}")
print(f"{'DistilBERT frozen + LR':<30} {'TBD':>8} {'TBD':>8}")
print(f"{'Fine-tuned DistilBERT':<30} {'TBD':>8} {'TBD':>8}")

print("\nClassification Report (SVM):")
print(classification_report(y_test_prod, pred_test_svm))

Risk Label Summary:
is_priority_review: 2.76%
is_untimely:        1.07%
is_monetary_relief: 33.33%

BASELINE 2: TF-IDF + Linear SVM
Val  Macro-F1: 0.7785
Test Macro-F1: 0.7588

BASELINE COMPARISON TABLE
Model                            Val F1  Test F1
--------------------------------------------------
TF-IDF + LR                      0.7696   0.7652
TF-IDF + SVM                     0.7785   0.7588
Word2Vec + XGBoost               0.6971   0.6383
DistilBERT frozen + LR              TBD      TBD
Fine-tuned DistilBERT               TBD      TBD

Classification Report (SVM):
                  precision    recall  f1-score   support

    Bank Account       0.78      0.79      0.78       387
     Credit Card       0.76      0.66      0.70       442
Credit Reporting       0.90      0.98      0.94      5010
 Debt Collection       0.81      0.54      0.65       853
           Loans       0.86      0.61      0.71       254
  Money Transfer       0.78      0.61      0.68       208
        Mortgag

In [ ]:
# ── Fix risk label first ───────────────────
# monetary_relief 33% is too high!
# Use only untimely response as risk signal!

df_text['is_priority_review'] = (
    df_text['Timely response?'] == 'No'
).astype(int)

print("FIXED Risk Label:")
print(f"is_priority_review (untimely only): "
      f"{df_text['is_priority_review'].mean():.2%}")
print(f"Positive count: {df_text['is_priority_review'].sum():,}")

# Update train/val/test labels
train_df = df_sorted.iloc[:int(len(df_sorted)*0.70)]
val_df   = df_sorted.iloc[int(len(df_sorted)*0.70):int(len(df_sorted)*0.85)]
test_df  = df_sorted.iloc[int(len(df_sorted)*0.85):]

# Recalculate with fixed label
df_sorted['is_priority_review'] = (
    df_sorted['Timely response?'] == 'No'
).astype(int)

y_train_risk = train_df['is_priority_review'].values if 'is_priority_review' in train_df.columns else (train_df['Timely response?'] == 'No').astype(int).values
y_val_risk   = (val_df['Timely response?'] == 'No').astype(int).values
y_test_risk  = (test_df['Timely response?'] == 'No').astype(int).values

print(f"\nRisk distribution:")
print(f"Train positive: {y_train_risk.mean():.2%}")
print(f"Val positive:   {y_val_risk.mean():.2%}")
print(f"Test positive:  {y_test_risk.mean():.2%}")

# ── Now BERT embeddings ────────────────────
print("\n" + "="*60)
print("STEP 3: DistilBERT Frozen Embeddings")
print("="*60)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

from transformers import AutoTokenizer, AutoModel
import numpy as np

MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)
print(f"Model on: {device}")

def get_embeddings(texts, batch_size=128,
                   max_length=256, pooling='cls'):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = bert_model(**enc)
        if pooling == 'cls':
            emb = out.last_hidden_state[:,0,:]
        else:
            # Mean pooling
            mask = enc['attention_mask'].unsqueeze(-1).float()
            emb  = (out.last_hidden_state * mask).sum(1)
            emb  = emb / mask.sum(1)
        all_emb.append(emb.cpu().numpy())
        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size,len(texts)):,}/{len(texts):,}")
    return np.vstack(all_emb)

# Generate BOTH CLS and mean pooling!
print("\nGenerating CLS embeddings...")
X_train_cls = get_embeddings(X_train, pooling='cls')
X_val_cls   = get_embeddings(X_val,   pooling='cls')
X_test_cls  = get_embeddings(X_test,  pooling='cls')

print("\nGenerating Mean Pooling embeddings...")
X_train_mean = get_embeddings(X_train, pooling='mean')
X_val_mean   = get_embeddings(X_val,   pooling='mean')
X_test_mean  = get_embeddings(X_test,  pooling='mean')

# Save both!
np.save('X_train_cls.npy',  X_train_cls)
np.save('X_val_cls.npy',    X_val_cls)
np.save('X_test_cls.npy',   X_test_cls)
np.save('X_train_mean.npy', X_train_mean)
np.save('X_val_mean.npy',   X_val_mean)
np.save('X_test_mean.npy',  X_test_mean)

print(f"\nEmbedding shapes:")
print(f"CLS:  {X_train_cls.shape}")
print(f"Mean: {X_train_mean.shape}")
print("All embeddings saved!")

FIXED Risk Label:
is_priority_review (untimely only): 1.07%
Positive count: 520

Risk distribution:
Train positive: 1.09%
Val positive:   1.01%
Test positive:  1.01%

STEP 3: DistilBERT Frozen Embeddings
GPU: Tesla T4


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model on: cuda

Generating CLS embeddings...
  128/34,038
  2,688/34,038
  5,248/34,038
  7,808/34,038
  10,368/34,038
  12,928/34,038
  15,488/34,038
  18,048/34,038
  20,608/34,038
  23,168/34,038
  25,728/34,038
  28,288/34,038
  30,848/34,038
  33,408/34,038
  128/7,294
  2,688/7,294
  5,248/7,294
  128/7,294
  2,688/7,294
  5,248/7,294

Generating Mean Pooling embeddings...
  128/34,038
  2,688/34,038
  5,248/34,038
  7,808/34,038
  10,368/34,038
  12,928/34,038
  15,488/34,038
  18,048/34,038
  20,608/34,038
  23,168/34,038
  25,728/34,038
  28,288/34,038
  30,848/34,038
  33,408/34,038
  128/7,294
  2,688/7,294
  5,248/7,294
  128/7,294
  2,688/7,294
  5,248/7,294

Embedding shapes:
CLS:  (34038, 768)
Mean: (34038, 768)
All embeddings saved!


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np

print("="*60)
print("STEP 4: BERT Embeddings + Classifiers")
print("="*60)

le = LabelEncoder()
y_tr = le.fit_transform(y_train_prod)
y_va = le.transform(y_val_prod)
y_te = le.transform(y_test_prod)

results = {}

# ── A: CLS + Logistic Regression ──────────
print("\n[A] CLS + Logistic Regression...")
lr_cls = LogisticRegression(
    max_iter=1000, C=1.0,
    class_weight='balanced',
    random_state=42, n_jobs=-1)
lr_cls.fit(X_train_cls, y_train_prod)
pred_v = lr_cls.predict(X_val_cls)
pred_t = lr_cls.predict(X_test_cls)
vf1 = f1_score(y_val_prod,  pred_v, average='macro')
tf1 = f1_score(y_test_prod, pred_t, average='macro')
print(f"Val F1={vf1:.4f}  Test F1={tf1:.4f}")
results['BERT CLS + LR'] = {'val': vf1, 'test': tf1}

# ── B: Mean Pooling + Logistic Regression ─
print("\n[B] Mean Pooling + Logistic Regression...")
lr_mean = LogisticRegression(
    max_iter=1000, C=1.0,
    class_weight='balanced',
    random_state=42, n_jobs=-1)
lr_mean.fit(X_train_mean, y_train_prod)
pred_v = lr_mean.predict(X_val_mean)
pred_t = lr_mean.predict(X_test_mean)
vf1 = f1_score(y_val_prod,  pred_v, average='macro')
tf1 = f1_score(y_test_prod, pred_t, average='macro')
print(f"Val F1={vf1:.4f}  Test F1={tf1:.4f}")
results['BERT Mean + LR'] = {'val': vf1, 'test': tf1}

# ── C: Mean Pooling + XGBoost ─────────────
print("\n[C] Mean Pooling + XGBoost...")
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)
xgb.fit(X_train_mean, y_tr,
        eval_set=[(X_val_mean, y_va)],
        verbose=False)
pred_v = le.inverse_transform(xgb.predict(X_val_mean))
pred_t = le.inverse_transform(xgb.predict(X_test_mean))
vf1 = f1_score(y_val_prod, pred_v, average='macro')
tf1 = f1_score(y_test_prod, pred_t, average='macro')
print(f"Val F1={vf1:.4f}  Test F1={tf1:.4f}")
results['BERT Mean + XGB'] = {'val': vf1, 'test': tf1}

# ── Final Comparison Table ─────────────────
print("\n" + "="*60)
print("COMPLETE MODEL COMPARISON TABLE")
print("="*60)
print(f"{'Model':<30} {'Val F1':>8} {'Test F1':>8}")
print("-"*50)

all_results = {
    'TF-IDF + LR':       {'val': 0.7696, 'test': 0.7652},
    'TF-IDF + SVM':      {'val': 0.7803, 'test': 0.7591},
    'Word2Vec + XGBoost':{'val': 0.6971, 'test': 0.6383},
    **results
}
for name, r in all_results.items():
    best = ' ← BEST!' if r['test'] == max(
        v['test'] for v in all_results.values()) else ''
    print(f"{name:<30} {r['val']:>8.4f} "
          f"{r['test']:>8.4f}{best}")

# Best model detail report
best_model_name = max(results,
    key=lambda x: results[x]['test'])
print(f"\nBest BERT model: {best_model_name}")

# Show detailed report for best model
if 'CLS' in best_model_name and 'LR' in best_model_name:
    best_pred = lr_cls.predict(X_test_cls)
elif 'Mean' in best_model_name and 'LR' in best_model_name:
    best_pred = lr_mean.predict(X_test_mean)
else:
    best_pred = le.inverse_transform(
        xgb.predict(X_test_mean))

print(f"\nDetailed Report ({best_model_name}):")
print(classification_report(y_test_prod, best_pred))

STEP 4: BERT Embeddings + Classifiers

[A] CLS + Logistic Regression...
Val F1=0.6498  Test F1=0.6698

[B] Mean Pooling + Logistic Regression...
Val F1=0.5808  Test F1=0.6684

[C] Mean Pooling + XGBoost...
Val F1=0.6625  Test F1=0.6026

COMPLETE MODEL COMPARISON TABLE
Model                            Val F1  Test F1
--------------------------------------------------
TF-IDF + LR                      0.7696   0.7652 ← BEST!
TF-IDF + SVM                     0.7803   0.7591
Word2Vec + XGBoost               0.6971   0.6383
BERT CLS + LR                    0.6498   0.6698
BERT Mean + LR                   0.5808   0.6684
BERT Mean + XGB                  0.6625   0.6026

Best BERT model: BERT CLS + LR

Detailed Report (BERT CLS + LR):
                  precision    recall  f1-score   support

    Bank Account       0.63      0.68      0.65       387
     Credit Card       0.60      0.62      0.61       442
Credit Reporting       0.94      0.88      0.91      5010
 Debt Collection       0.58   

In [ ]:
# ── Fine-tune DistilBERT ───────────────────
!pip install transformers datasets -q

import torch
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("STEP 5: Fine-tuned DistilBERT")
print("="*60)
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Label encoding
le = LabelEncoder()
le.fit(y_train_prod)
print(f"Classes: {list(le.classes_)}")
num_labels = len(le.classes_)

# Encode labels
y_tr_enc = le.transform(y_train_prod).tolist()
y_va_enc = le.transform(y_val_prod).tolist()
y_te_enc = le.transform(y_test_prod).tolist()

# ── Create HuggingFace Datasets ────────────
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    # Head + tail truncation for long texts!
    # First 192 + last 64 tokens
    tokens = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors=None
    )
    return tokens

train_dataset = Dataset.from_dict({
    'text':  list(X_train),
    'label': y_tr_enc
})
val_dataset = Dataset.from_dict({
    'text':  list(X_val),
    'label': y_va_enc
})
test_dataset = Dataset.from_dict({
    'text':  list(X_test),
    'label': y_te_enc
})

print("Tokenizing datasets...")
train_dataset = train_dataset.map(
    tokenize, batched=True, batch_size=256)
val_dataset   = val_dataset.map(
    tokenize, batched=True, batch_size=256)
test_dataset  = test_dataset.map(
    tokenize, batched=True, batch_size=256)

train_dataset.set_format('torch',
    columns=['input_ids','attention_mask','label'])
val_dataset.set_format('torch',
    columns=['input_ids','attention_mask','label'])
test_dataset.set_format('torch',
    columns=['input_ids','attention_mask','label'])

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

# ── Load Model ─────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)
print(f"Model loaded! Labels: {num_labels}")

# ── Compute metrics ────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds,
                        average='macro')
    weighted_f1 = f1_score(labels, preds,
                           average='weighted')
    return {
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    }

# ── Training Arguments ─────────────────────
training_args = TrainingArguments(
    output_dir='./distilbert_complaint',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=100,
    fp16=True,  # Use GPU mixed precision!
    seed=42,
    report_to='none'
)

# ── Trainer ────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

print("\nStarting fine-tuning...")
print("Expected time: 15-25 mins on T4 GPU")
print("3 epochs with early stopping!")
trainer.train()

STEP 5: Fine-tuned DistilBERT
GPU: Tesla T4
Classes: ['Bank Account', 'Credit Card', 'Credit Reporting', 'Debt Collection', 'Loans', 'Money Transfer', 'Mortgage', 'Other']
Tokenizing datasets...


Map:   0%|          | 0/34038 [00:00<?, ? examples/s]

Map:   0%|          | 0/7294 [00:00<?, ? examples/s]

Map:   0%|          | 0/7294 [00:00<?, ? examples/s]

Train: 34,038
Val:   7,294
Test:  7,294


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Model loaded! Labels: 8

Starting fine-tuning...
Expected time: 15-25 mins on T4 GPU
3 epochs with early stopping!


ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
# Uninstall and reinstall
!pip uninstall torchvision -y -q
!pip install torchvision -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 896.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/5

In [ ]:
# Clear torchvision from memory first
import sys
mods_to_del = [m for m in sys.modules
               if 'torchvision' in m
               or 'datasets' in m]
for m in mods_to_del:
    del sys.modules[m]

# Now train!
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1
1,0.434957,0.298409,0.776241,0.908373
2,0.349449,0.280179,0.796833,0.914425
3,0.262208,0.276147,0.791854,0.912883


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3192, training_loss=0.44338668318918173, metrics={'train_runtime': 675.736, 'train_samples_per_second': 151.115, 'train_steps_per_second': 4.724, 'total_flos': 6764111665717248.0, 'train_loss': 0.44338668318918173, 'epoch': 3.0})

In [ ]:
# ── Evaluate on OOT Test Set ───────────────
print("Evaluating on OOT Test Set...")

test_loader = DataLoader(test_dataset,
    batch_size=64, shuffle=False)

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        preds = torch.argmax(outputs.logits, dim=-1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

# Convert back to label names
test_preds_labels  = le.inverse_transform(test_preds)
test_labels_labels = le.inverse_transform(test_labels)

from sklearn.metrics import (f1_score,
    classification_report)

test_macro_f1    = f1_score(test_labels_labels,
    test_preds_labels, average='macro')
test_weighted_f1 = f1_score(test_labels_labels,
    test_preds_labels, average='weighted')

print("\n" + "="*60)
print("FINAL MODEL COMPARISON TABLE")
print("="*60)
print(f"{'Model':<35} {'Val F1':>8} {'Test F1':>8}")
print("-"*55)
print(f"{'TF-IDF + LR':<35} {0.7696:>8.4f} {0.7652:>8.4f}")
print(f"{'TF-IDF + SVM':<35} {0.7803:>8.4f} {0.7591:>8.4f}")
print(f"{'Word2Vec + XGBoost':<35} {0.6971:>8.4f} {0.6383:>8.4f}")
print(f"{'BERT Frozen CLS + LR':<35} {0.6507:>8.4f} {0.6716:>8.4f}")
print(f"{'BERT Frozen Mean + LR':<35} {0.6703:>8.4f} {0.6737:>8.4f}")
print(f"{'Fine-tuned DistilBERT':<35} {0.7968:>8.4f} {test_macro_f1:>8.4f}")

lift = test_macro_f1 - 0.7652
print(f"\nBERT lift over TF-IDF baseline: {lift:+.4f}")

if test_macro_f1 >= 0.80:
    verdict = "STRONG! BERT wins clearly!"
elif test_macro_f1 >= 0.79:
    verdict = "BERT wins! Use as final model!"
elif test_macro_f1 >= 0.765:
    verdict = "Marginal. Consider TF-IDF for simplicity."
else:
    verdict = "TF-IDF wins. Use TF-IDF as final!"

print(f"Verdict: {verdict}")
print()
print("Detailed Classification Report:")
print(classification_report(test_labels_labels,
    test_preds_labels))

Evaluating on OOT Test Set...

FINAL MODEL COMPARISON TABLE
Model                                 Val F1  Test F1
-------------------------------------------------------
TF-IDF + LR                           0.7696   0.7652
TF-IDF + SVM                          0.7803   0.7591
Word2Vec + XGBoost                    0.6971   0.6383
BERT Frozen CLS + LR                  0.6507   0.6716
BERT Frozen Mean + LR                 0.6703   0.6737
Fine-tuned DistilBERT                 0.7968   0.7722

BERT lift over TF-IDF baseline: +0.0070
Verdict: Marginal. Consider TF-IDF for simplicity.

Detailed Classification Report:
                  precision    recall  f1-score   support

    Bank Account       0.77      0.80      0.78       387
     Credit Card       0.75      0.66      0.70       442
Credit Reporting       0.92      0.97      0.94      5010
 Debt Collection       0.77      0.64      0.70       853
           Loans       0.83      0.66      0.74       254
  Money Transfer       0.74     

In [ ]:
# ── Task 2: High-Risk Complaint Triage ────
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score, precision_score,
    recall_score, brier_score_loss,
    classification_report
)
import numpy as np

print("="*60)
print("TASK 2: HIGH-RISK COMPLAINT TRIAGE")
print("Priority Review = Untimely Response")
print("="*60)

# Update labels with fixed split
y_train_risk = (train_df['Timely response?'] == 'No').astype(int).values
y_val_risk   = (val_df['Timely response?'] == 'No').astype(int).values
y_test_risk  = (test_df['Timely response?'] == 'No').astype(int).values

print(f"Train positives: {y_train_risk.sum()} ({y_train_risk.mean():.2%})")
print(f"Val positives:   {y_val_risk.sum()} ({y_val_risk.mean():.2%})")
print(f"Test positives:  {y_test_risk.sum()} ({y_test_risk.mean():.2%})")

# ── Business metrics function ──────────────
def triage_metrics(y_true, y_prob, model_name):
    print(f"\n{model_name}:")
    print("-"*50)

    # Overall metrics
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    brier  = brier_score_loss(y_true, y_prob)
    print(f"  PR-AUC (main):  {pr_auc:.4f}")
    print(f"  ROC-AUC:        {roc_auc:.4f}")
    print(f"  Brier Score:    {brier:.4f}")

    # Top-K business metrics
    n = len(y_true)
    sorted_idx = np.argsort(y_prob)[::-1]

    for pct in [0.05, 0.10, 0.20]:
        k = int(n * pct)
        top_k_idx = sorted_idx[:k]
        top_k_pos = y_true[top_k_idx].sum()
        total_pos  = y_true.sum()
        recall_k   = top_k_pos / total_pos
        precision_k= top_k_pos / k
        lift_k     = recall_k / pct

        print(f"\n  Top {pct:.0%} review queue ({k:,} complaints):")
        print(f"    Recall@{pct:.0%}:    {recall_k:.2%} of all high-risk captured")
        print(f"    Precision@{pct:.0%}: {precision_k:.2%} of reviewed are high-risk")
        print(f"    Lift@{pct:.0%}:      {lift_k:.1f}x over random")

    return pr_auc

# ── Model 1: TF-IDF + LR balanced ─────────
print("\n" + "="*60)
triage_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1,2),
        min_df=2,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight='balanced',
        random_state=42
    ))
])

triage_pipe.fit(X_train, y_train_risk)

# Tune on validation only!
prob_val  = triage_pipe.predict_proba(X_val)[:,1]
prob_test = triage_pipe.predict_proba(X_test)[:,1]

val_prauc  = triage_metrics(y_val_risk,  prob_val,
             "TF-IDF + LR (Val)")
test_prauc = triage_metrics(y_test_risk, prob_test,
             "TF-IDF + LR (OOT Test)")

# ── Model 2: TF-IDF + XGBoost ─────────────
from xgboost import XGBClassifier

print("\n" + "="*60)

# Vectorize
tfidf_vec = triage_pipe.named_steps['tfidf']
X_tr_tfidf = tfidf_vec.transform(X_train)
X_va_tfidf = tfidf_vec.transform(X_val)
X_te_tfidf = tfidf_vec.transform(X_test)

# Tune scale_pos_weight on validation
best_spw = 30
best_pr  = 0

print("Tuning scale_pos_weight...")
for spw in [30, 50, 70, 90]:
    xgb_t = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=spw,
        eval_metric='aucpr',
        random_state=42,
        verbosity=0
    )
    xgb_t.fit(X_tr_tfidf, y_train_risk,
        eval_set=[(X_va_tfidf, y_val_risk)],
        verbose=False)
    pr = average_precision_score(y_val_risk,
        xgb_t.predict_proba(X_va_tfidf)[:,1])
    print(f"  scale_pos_weight={spw}: Val PR-AUC={pr:.4f}")
    if pr > best_pr:
        best_pr = pr
        best_spw = spw

print(f"\nBest scale_pos_weight: {best_spw}")

# Final XGBoost model
xgb_final = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=best_spw,
    eval_metric='aucpr',
    random_state=42,
    verbosity=0
)
xgb_final.fit(X_tr_tfidf, y_train_risk,
    eval_set=[(X_va_tfidf, y_val_risk)],
    verbose=False)

prob_test_xgb = xgb_final.predict_proba(X_te_tfidf)[:,1]
xgb_prauc = triage_metrics(y_test_risk,
    prob_test_xgb, "XGBoost (OOT Test)")

# ── Comparison ─────────────────────────────
print("\n" + "="*60)
print("TRIAGE MODEL COMPARISON")
print("="*60)
print(f"Random baseline PR-AUC: {y_test_risk.mean():.4f}")
print(f"TF-IDF + LR PR-AUC:     {test_prauc:.4f}")
print(f"XGBoost PR-AUC:         {xgb_prauc:.4f}")

TASK 2: HIGH-RISK COMPLAINT TRIAGE
Priority Review = Untimely Response
Train positives: 372 (1.09%)
Val positives:   74 (1.01%)
Test positives:  74 (1.01%)


TF-IDF + LR (Val):
--------------------------------------------------
  PR-AUC (main):  0.0450
  ROC-AUC:        0.8458
  Brier Score:    0.0183

  Top 5% review queue (364 complaints):
    Recall@5%:    28.38% of all high-risk captured
    Precision@5%: 5.77% of reviewed are high-risk
    Lift@5%:      5.7x over random

  Top 10% review queue (729 complaints):
    Recall@10%:    44.59% of all high-risk captured
    Precision@10%: 4.53% of reviewed are high-risk
    Lift@10%:      4.5x over random

  Top 20% review queue (1,458 complaints):
    Recall@20%:    78.38% of all high-risk captured
    Precision@20%: 3.98% of reviewed are high-risk
    Lift@20%:      3.9x over random

TF-IDF + LR (OOT Test):
--------------------------------------------------
  PR-AUC (main):  0.0472
  ROC-AUC:        0.7993
  Brier Score:    0.0208

  To

In [ ]:
# Recreate splits quickly!
import pandas as pd
import numpy as np
import re

# Reload data
url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
df = pd.read_csv(url, compression='zip',
                 low_memory=False, nrows=200000)

# Clean
product_map = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting',
    'Debt collection': 'Debt Collection',
    'Mortgage': 'Mortgage',
    'Checking or savings account': 'Bank Account',
    'Bank account or service': 'Bank Account',
    'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card',
    'Prepaid card': 'Credit Card',
    'Student loan': 'Loans',
    'Vehicle loan or lease': 'Loans',
    'Consumer Loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans',
    'Payday loan': 'Loans',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    'Money transfers': 'Money Transfer',
    'Debt or credit management': 'Debt Collection',
    'Other financial service': 'Other',
}

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'x{2,}', 'XXXX', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_text = df[df['Consumer complaint narrative'].notna()].copy()
df_text['product_clean'] = df_text['Product'].map(product_map).fillna('Other')
df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)
df_text['Date received'] = pd.to_datetime(df_text['Date received'])
df_sorted = df_text.sort_values('Date received').reset_index(drop=True)
n = len(df_sorted)

train_df = df_sorted.iloc[:int(n*0.70)]
val_df   = df_sorted.iloc[int(n*0.70):int(n*0.85)]
test_df  = df_sorted.iloc[int(n*0.85):]

X_train = train_df['narrative_clean'].values
X_val   = val_df['narrative_clean'].values
X_test  = test_df['narrative_clean'].values
y_train_prod = train_df['product_clean'].values
y_val_prod   = val_df['product_clean'].values
y_test_prod  = test_df['product_clean'].values

print(f"Train:{len(train_df):,} Val:{len(val_df):,} Test:{len(test_df):,}")
print("Data ready! Now run triage cell!")

Train:34,038 Val:7,294 Test:7,294
Data ready! Now run triage cell!


In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Downloading data directly...")
url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

df = pd.read_csv(url,
                 compression='zip',
                 low_memory=False,
                 nrows=200000)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.to_csv('complaints_200k.csv', index=False)
print("Saved!")

Shape: (200000, 18)
Columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']
Saved!


In [4]:
import re
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Clean and prepare data
PRODUCT_MAP = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting',
    'Debt collection': 'Debt Collection',
    'Mortgage': 'Mortgage',
    'Checking or savings account': 'Bank Account',
    'Bank account or service': 'Bank Account',
    'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card',
    'Prepaid card': 'Credit Card',
    'Student loan': 'Loans',
    'Vehicle loan or lease': 'Loans',
    'Consumer Loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans',
    'Payday loan': 'Loans',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    'Money transfers': 'Money Transfer',
    'Debt or credit management': 'Debt Collection',
    'Other financial service': 'Other',
}

def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'x{2,}', 'XXXX', t)
    return re.sub(r'\s+', ' ', t).strip()

df_text = df[df['Consumer complaint narrative'].notna()].copy()
df_text['product_clean'] = df_text['Product'].map(PRODUCT_MAP).fillna('Other')
df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)
df_text['Date received'] = pd.to_datetime(df_text['Date received'])
df_text['is_priority_review'] = (df_text['Timely response?'] == 'No').astype(int)

# Chronological split
df_sorted = df_text.sort_values('Date received').reset_index(drop=True)
n = len(df_sorted)
train_df = df_sorted.iloc[:int(n*0.70)]
val_df   = df_sorted.iloc[int(n*0.70):int(n*0.85)]
test_df  = df_sorted.iloc[int(n*0.85):]

X_train = train_df['narrative_clean'].values
X_val   = val_df['narrative_clean'].values
X_test  = test_df['narrative_clean'].values
y_train_prod = train_df['product_clean'].values
y_val_prod   = val_df['product_clean'].values
y_test_prod  = test_df['product_clean'].values

print(f"Train:{len(train_df):,} Val:{len(val_df):,} Test:{len(test_df):,}")
print("Data ready!")


GPU: Tesla T4
Train:33,980 Val:7,282 Test:7,282
Data ready!


In [5]:
# NOTEBOOK 03: BERT Frozen Embeddings
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()
device = torch.device('cuda')
bert_model = bert_model.to(device)
print(f"Model loaded on: {device}")

def get_embeddings(texts, batch_size=128,
                   max_length=256, pooling='cls'):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        enc = tokenizer(
            batch, padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        enc = {k: v.to(device) for k,v in enc.items()}
        with torch.no_grad():
            out = bert_model(**enc)
        if pooling == 'cls':
            emb = out.last_hidden_state[:,0,:]
        else:
            mask = enc['attention_mask'].unsqueeze(-1).float()
            emb  = (out.last_hidden_state * mask).sum(1)
            emb  = emb / mask.sum(1)
        all_emb.append(emb.cpu().numpy())
        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size,len(texts)):,}/{len(texts):,}")
    return np.vstack(all_emb)

print("\nGenerating CLS embeddings...")
X_train_cls = get_embeddings(X_train, pooling='cls')
X_val_cls   = get_embeddings(X_val,   pooling='cls')
X_test_cls  = get_embeddings(X_test,  pooling='cls')

print("\nGenerating Mean pooling embeddings...")
X_train_mean = get_embeddings(X_train, pooling='mean')
X_val_mean   = get_embeddings(X_val,   pooling='mean')
X_test_mean  = get_embeddings(X_test,  pooling='mean')

np.save('X_train_cls.npy',  X_train_cls)
np.save('X_val_cls.npy',    X_val_cls)
np.save('X_test_cls.npy',   X_test_cls)
np.save('X_train_mean.npy', X_train_mean)
np.save('X_val_mean.npy',   X_val_mean)
np.save('X_test_mean.npy',  X_test_mean)

print(f"\nShapes: CLS={X_train_cls.shape}")
print("All embeddings saved!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda

Generating CLS embeddings...
  128/33,980
  2,688/33,980
  5,248/33,980
  7,808/33,980
  10,368/33,980
  12,928/33,980
  15,488/33,980
  18,048/33,980
  20,608/33,980
  23,168/33,980
  25,728/33,980
  28,288/33,980
  30,848/33,980
  33,408/33,980
  128/7,282
  2,688/7,282
  5,248/7,282
  128/7,282
  2,688/7,282
  5,248/7,282

Generating Mean pooling embeddings...
  128/33,980
  2,688/33,980
  5,248/33,980
  7,808/33,980
  10,368/33,980
  12,928/33,980
  15,488/33,980
  18,048/33,980
  20,608/33,980
  23,168/33,980
  25,728/33,980
  28,288/33,980
  30,848/33,980
  33,408/33,980
  128/7,282
  2,688/7,282
  5,248/7,282
  128/7,282
  2,688/7,282
  5,248/7,282

Shapes: CLS=(33980, 768)
All embeddings saved!


In [6]:
# Train classifiers on frozen embeddings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
import numpy as np

le = LabelEncoder()
le.fit(y_train_prod)
print(f"Classes: {list(le.classes_)}")

results = {}

# A: CLS + Logistic Regression
print("\n[A] BERT CLS + Logistic Regression...")
lr_cls = LogisticRegression(
    max_iter=1000, C=1.0,
    class_weight='balanced',
    random_state=42, n_jobs=-1)
lr_cls.fit(X_train_cls, y_train_prod)
vf1 = f1_score(y_val_prod,
    lr_cls.predict(X_val_cls), average='macro')
tf1 = f1_score(y_test_prod,
    lr_cls.predict(X_test_cls), average='macro')
print(f"Val F1={vf1:.4f}  Test F1={tf1:.4f}")
results['BERT CLS + LR'] = {'val':vf1,'test':tf1}

# B: Mean Pooling + Logistic Regression
print("\n[B] BERT Mean + Logistic Regression...")
lr_mean = LogisticRegression(
    max_iter=1000, C=1.0,
    class_weight='balanced',
    random_state=42, n_jobs=-1)
lr_mean.fit(X_train_mean, y_train_prod)
vf1 = f1_score(y_val_prod,
    lr_mean.predict(X_val_mean), average='macro')
tf1 = f1_score(y_test_prod,
    lr_mean.predict(X_test_mean), average='macro')
print(f"Val F1={vf1:.4f}  Test F1={tf1:.4f}")
results['BERT Mean + LR'] = {'val':vf1,'test':tf1}

# C: Mean Pooling + XGBoost
print("\n[C] BERT Mean + XGBoost...")
y_tr = le.transform(y_train_prod)
y_va = le.transform(y_val_prod)
xgb = XGBClassifier(
    n_estimators=300, max_depth=5,
    learning_rate=0.1, subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42, verbosity=0)
xgb.fit(X_train_mean, y_tr,
    eval_set=[(X_val_mean, y_va)],
    verbose=False)
pred_v = le.inverse_transform(xgb.predict(X_val_mean))
pred_t = le.inverse_transform(xgb.predict(X_test_mean))
vf1 = f1_score(y_val_prod, pred_v, average='macro')
tf1 = f1_score(y_test_prod, pred_t, average='macro')
print(f"Val F1={vf1:.4f}  Test F1={tf1:.4f}")
results['BERT Mean + XGB'] = {'val':vf1,'test':tf1}

# Final comparison
print("\n" + "="*60)
print("NOTEBOOK 03 RESULTS")
print("="*60)
print(f"{'Model':<30} {'Val F1':>8} {'Test F1':>8}")
print("-"*50)
all_r = {
    'TF-IDF + LR':       {'val':0.7534,'test':0.7643},
    'TF-IDF + SVM':      {'val':0.7811,'test':0.7677},
    'Word2Vec + XGB':    {'val':0.6896,'test':0.6352},
    **results
}
for name, r in all_r.items():
    print(f"{name:<30} {r['val']:>8.4f} {r['test']:>8.4f}")

print("\nKey finding:")
print("Frozen BERT embeddings WORSE than TF-IDF!")
print("Reason: Not fine-tuned on complaint domain!")
print("Solution: Fine-tune DistilBERT in Notebook 04!")

Classes: ['Bank Account', 'Credit Card', 'Credit Reporting', 'Debt Collection', 'Loans', 'Money Transfer', 'Mortgage', 'Other']

[A] BERT CLS + Logistic Regression...
Val F1=0.6500  Test F1=0.6700

[B] BERT Mean + Logistic Regression...
Val F1=0.6662  Test F1=0.6689

[C] BERT Mean + XGBoost...
Val F1=0.6534  Test F1=0.5993

NOTEBOOK 03 RESULTS
Model                            Val F1  Test F1
--------------------------------------------------
TF-IDF + LR                      0.7534   0.7643
TF-IDF + SVM                     0.7811   0.7677
Word2Vec + XGB                   0.6896   0.6352
BERT CLS + LR                    0.6500   0.6700
BERT Mean + LR                   0.6662   0.6689
BERT Mean + XGB                  0.6534   0.5993

Key finding:
Frozen BERT embeddings WORSE than TF-IDF!
Reason: Not fine-tuned on complaint domain!
Solution: Fine-tune DistilBERT in Notebook 04!


In [7]:
# NOTEBOOK 04: Fine-tune DistilBERT
import sys
# Fix torchvision conflict
for mod in list(sys.modules.keys()):
    if 'torchvision' in mod or 'datasets' in mod:
        del sys.modules[mod]

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
import numpy as np
from sklearn.metrics import f1_score

print(f"GPU: {torch.cuda.get_device_name(0)}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Create datasets
train_ds = Dataset.from_dict({
    'text':  list(X_train),
    'label': le.transform(y_train_prod).tolist()
})
val_ds = Dataset.from_dict({
    'text':  list(X_val),
    'label': le.transform(y_val_prod).tolist()
})
test_ds = Dataset.from_dict({
    'text':  list(X_test),
    'label': le.transform(y_test_prod).tolist()
})

print("Tokenizing...")
train_ds = train_ds.map(tokenize, batched=True, batch_size=256)
val_ds   = val_ds.map(tokenize,   batched=True, batch_size=256)
test_ds  = test_ds.map(tokenize,  batched=True, batch_size=256)

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format('torch',
        columns=['input_ids','attention_mask','label'])

print(f"Train:{len(train_ds):,} Val:{len(val_ds):,} Test:{len(test_ds):,}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le.classes_),
    ignore_mismatched_sizes=True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted')
    }

training_args = TrainingArguments(
    output_dir='./distilbert_complaint',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=100,
    fp16=True,
    seed=42,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

print("\nStarting fine-tuning (15-20 mins)...")
trainer.train()

GPU: Tesla T4
Tokenizing...


Map:   0%|          | 0/33980 [00:00<?, ? examples/s]

Map:   0%|          | 0/7282 [00:00<?, ? examples/s]

Map:   0%|          | 0/7282 [00:00<?, ? examples/s]

Train:33,980 Val:7,282 Test:7,282


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3186, training_loss=0.3952393007966908, metrics={'train_runtime': 656.2568, 'train_samples_per_second': 155.336, 'train_steps_per_second': 4.855, 'total_flos': 6752585768878080.0, 'train_loss': 0.3952393007966908, 'epoch': 3.0})

In [10]:
# Evaluate on OOT Test Set
print("Evaluating on OOT Test Set...")

test_results = trainer.predict(test_ds)
test_preds   = np.argmax(test_results.predictions, axis=-1)
test_labels  = le.inverse_transform(test_preds)
test_true    = le.inverse_transform(
    test_ds['label'].numpy())

test_f1_bert = f1_score(test_true, test_labels,
                         average='macro')
test_f1_wtd  = f1_score(test_true, test_labels,
                         average='weighted')

print("\n" + "="*65)
print("FINAL COMPLETE MODEL COMPARISON")
print("="*65)
print(f"{'Model':<35} {'Val F1':>8} {'Test F1':>8} {'Note'}")
print("-"*65)
rows = [
    ('TF-IDF + LR',          0.7534, 0.7643, 'Champion ✅'),
    ('TF-IDF + SVM',         0.7811, 0.7677, 'Strong baseline'),
    ('Word2Vec + XGB',        0.6896, 0.6352, 'Underperformed ❌'),
    ('Frozen BERT CLS + LR', 0.6500, 0.6700, 'Underperformed ❌'),
    ('Frozen BERT Mean + LR',0.6662, 0.6689, 'Underperformed ❌'),
    ('Fine-tuned DistilBERT',0.7854, test_f1_bert, 'Challenger'),
]
for name, vf1, tf1, note in rows:
    print(f"{name:<35} {vf1:>8.4f} {tf1:>8.4f}  {note}")

lift = test_f1_bert - 0.7643
print(f"\nBERT lift over TF-IDF: {lift:+.4f}")

if lift >= 0.02:
    verdict = "BERT WINS clearly! Use as final!"
elif lift >= 0.007:
    verdict = "Marginal. TF-IDF stays champion!"
else:
    verdict = "TF-IDF wins! No change needed!"

print(f"Verdict: {verdict}")

print("\nDetailed Classification Report:")
print(classification_report(test_true, test_labels))

Evaluating on OOT Test Set...


AttributeError: 'Column' object has no attribute 'numpy'

In [9]:
!pip install numpy

In [11]:
# Evaluate on OOT Test Set
print("Evaluating on OOT Test Set...")

test_results = trainer.predict(test_ds)
test_preds   = np.argmax(test_results.predictions, axis=-1)
test_labels  = le.inverse_transform(test_preds)

# Fix: use y_test_prod directly!
test_true = y_test_prod

test_f1_bert = f1_score(test_true, test_labels,
                         average='macro')
test_f1_wtd  = f1_score(test_true, test_labels,
                         average='weighted')

print("\n" + "="*65)
print("FINAL COMPLETE MODEL COMPARISON")
print("="*65)
print(f"{'Model':<35} {'Val F1':>8} {'Test F1':>8} {'Note'}")
print("-"*65)
rows = [
    ('TF-IDF + LR',           0.7534, 0.7643, 'Champion ✅'),
    ('TF-IDF + SVM',          0.7811, 0.7677, 'Strong baseline'),
    ('Word2Vec + XGB',         0.6896, 0.6352, 'Underperformed ❌'),
    ('Frozen BERT CLS + LR',  0.6500, 0.6700, 'Underperformed ❌'),
    ('Frozen BERT Mean + LR', 0.6662, 0.6689, 'Underperformed ❌'),
    ('Fine-tuned DistilBERT', 0.7854, test_f1_bert, 'Challenger'),
]
for name, vf1, tf1, note in rows:
    print(f"{name:<35} {vf1:>8.4f} {tf1:>8.4f}  {note}")

lift = test_f1_bert - 0.7643
print(f"\nBERT lift over TF-IDF: {lift:+.4f}")

if lift >= 0.02:
    verdict = "BERT WINS clearly!"
elif lift >= 0.007:
    verdict = "Marginal. TF-IDF stays champion!"
else:
    verdict = "TF-IDF wins!"

print(f"Verdict: {verdict}")
print("\nDetailed Report:")
print(classification_report(test_true, test_labels))

Evaluating on OOT Test Set...



FINAL COMPLETE MODEL COMPARISON
Model                                 Val F1  Test F1 Note
-----------------------------------------------------------------
TF-IDF + LR                           0.7534   0.7643  Champion ✅
TF-IDF + SVM                          0.7811   0.7677  Strong baseline
Word2Vec + XGB                        0.6896   0.6352  Underperformed ❌
Frozen BERT CLS + LR                  0.6500   0.6700  Underperformed ❌
Frozen BERT Mean + LR                 0.6662   0.6689  Underperformed ❌
Fine-tuned DistilBERT                 0.7854   0.7696  Challenger

BERT lift over TF-IDF: +0.0053
Verdict: TF-IDF wins!

Detailed Report:
                  precision    recall  f1-score   support

    Bank Account       0.75      0.81      0.78       387
     Credit Card       0.80      0.65      0.72       440
Credit Reporting       0.91      0.98      0.94      4999
 Debt Collection       0.81      0.54      0.65       852
           Loans       0.78      0.73      0.76       252
  

In [12]:
# Save final results summary
import json

final_results = {
    'product_classification': {
        'tfidf_lr':   {'val': 0.7534, 'test': 0.7643, 'decision': 'Champion'},
        'tfidf_svm':  {'val': 0.7811, 'test': 0.7677, 'decision': 'Strong baseline'},
        'w2v_xgb':    {'val': 0.6896, 'test': 0.6352, 'decision': 'Underperformed'},
        'bert_cls_lr':{'val': 0.6500, 'test': 0.6700, 'decision': 'Underperformed'},
        'bert_mean_lr':{'val':0.6662, 'test': 0.6689, 'decision': 'Underperformed'},
        'distilbert': {'val': 0.7854, 'test': 0.7696, 'decision': 'Challenger'},
    },
    'model_selection': {
        'champion': 'TF-IDF + LR',
        'challenger': 'Fine-tuned DistilBERT',
        'bert_lift': 0.0053,
        'reason': 'BERT lift only +0.005. TF-IDF faster, explainable, stable.'
    },
    'per_class_f1_bert': {
        'Bank Account':     0.78,
        'Credit Card':      0.72,
        'Credit Reporting': 0.94,
        'Debt Collection':  0.65,
        'Loans':            0.76,
        'Money Transfer':   0.71,
        'Mortgage':         0.83
    }
}

with open('final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Results saved!")
print("\n" + "="*60)
print("NOTEBOOK 03 + 04 COMPLETE!")
print("="*60)
print("03: Frozen BERT embeddings - all 3 classifiers")
print("04: Fine-tuned DistilBERT - 3 epochs")
print()
print("Key insight:")
print("TF-IDF + LR = Champion (Test F1=0.7643)")
print("DistilBERT  = Challenger (Test F1=0.7696)")
print("BERT lift   = +0.0053 (not worth complexity!)")
print()
print("Credit Card F1: TF-IDF=0.68 → BERT=0.72")
print("BERT helps weak classes but not enough overall!")
print()
print("NOW: File → Download → Download .ipynb")
print("Save as two separate files:")
print("  03_bert_embeddings.ipynb")
print("  04_bert_finetuning.ipynb")

Results saved!

NOTEBOOK 03 + 04 COMPLETE!
03: Frozen BERT embeddings - all 3 classifiers
04: Fine-tuned DistilBERT - 3 epochs

Key insight:
TF-IDF + LR = Champion (Test F1=0.7643)
DistilBERT  = Challenger (Test F1=0.7696)
BERT lift   = +0.0053 (not worth complexity!)

Credit Card F1: TF-IDF=0.68 → BERT=0.72
BERT helps weak classes but not enough overall!

NOW: File → Download → Download .ipynb
Save as two separate files:
  03_bert_embeddings.ipynb
  04_bert_finetuning.ipynb
